<a href="https://colab.research.google.com/github/csu-techhub/quantum-optimization-simulation/blob/main/Module3_Labs/Lab10.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/> </a>

# Lab 10 — Measuring Pauli Terms and Estimating Energy
**Quantum Optimization and Simulation — VQE Laboratory Series**

Prepare a simple two-qubit H₂-like trial state, measure Pauli terms, and reconstruct its energy.

**Suggested use:** brief instructor demonstration followed by guided or independent work.

**Notebook style:** Most code is supplied. The main idea is that an energy is assembled from several measurement settings.

> Qiskit displays measured bitstrings as `q_(n-1)...q_0`.


## Learning objectives
- Prepare a two-qubit parameterized trial state.
- Measure \(Z_0\), \(Z_1\), and \(Z_0Z_1\) from one Z-basis data set.
- Measure \(X_0X_1\) and \(Y_0Y_1\) using basis changes.
- Reconstruct an energy from Pauli expectation values.
- Count how many measurement circuits are executed.


In [ ]:
# Run once in a fresh Google Colab session.
%pip -q install pylatexenc matplotlib
%pip -q install "qiskit~=2.5" "qiskit-aer~=0.17" "qiskit-algorithms~=0.4" "qiskit-nature~=0.8"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit, transpile
from qiskit.visualization import plot_histogram
from qiskit.quantum_info import Statevector, SparsePauliOp
from qiskit_aer import AerSimulator

SEED = 123
SHOTS = 4096

CIRCUIT_EXECUTIONS = 0

def reset_execution_counter():
    global CIRCUIT_EXECUTIONS
    CIRCUIT_EXECUTIONS = 0

def run_counts(qc, shots=SHOTS, noise_model=None, seed=SEED):
    global CIRCUIT_EXECUTIONS
    backend = AerSimulator(noise_model=noise_model)
    tqc = transpile(qc, backend, optimization_level=1)
    result = backend.run(tqc, shots=shots, seed_simulator=seed).result()
    CIRCUIT_EXECUTIONS += 1
    return result.get_counts()

def q0_first(qiskit_bits):
    return qiskit_bits.replace(" ", "")[::-1]


## Part A — Prepare a two-qubit state

In [ ]:
theta = 0.45

qc_state = QuantumCircuit(2)
qc_state.x(0)            # Start in |01> in Qiskit's |q1 q0> display
qc_state.ry(2*theta, 1)
qc_state.cx(1, 0)

display(qc_state.draw("mpl"))
sv = Statevector.from_instruction(qc_state)
print(np.round(sv.data, 3))

##Check:
Two-qubit statevector $\begin{pmatrix} c_{00} \\ c_{01} \\ c_{10} \\ c_{11} \end{pmatrix} = \begin{pmatrix} 0 \\ 0.900 \\ 0.435 \\ 0 \end{pmatrix}$ represents a quantum superposition across the 4 computational basis states $\{\vert{}00\rangle, \vert{}01\rangle, \vert{}10\rangle, \vert{}11\rangle\}$ (in Qiskit's $\vert{}q_1 q_0\rangle$ convention).

The statevector corresponds to the physical state:$$\vert{}\psi\rangle = 0.900\,\vert{}01\rangle + 0.435\,\vert{}10\rangle$$

* Zero probability for $\vert{}00\rangle$ and $\vert{}11\rangle$: The qubits will never both be $0$ or both be $1$.
* Single-excitation state: Exactly one qubit is set to $\vert{}1\rangle$ at any given time.

## Helper functions

In [ ]:
def parity_expectation(counts, qubits):
    total = sum(counts.values())
    value = 0.0

    for bits, count in counts.items():
        bits = bits.replace(" ", "")
        # Qiskit string is q_(n-1)...q_0
        eigenvalue = 1
        for q in qubits:
            bit = int(bits[-1-q])
            eigenvalue *= (1 if bit == 0 else -1)
        value += eigenvalue * count / total
    return value

def measure_pauli_pair(base_circuit, basis):
    qc = base_circuit.copy()

    if basis == "X":
        qc.h([0,1])
    elif basis == "Y":
        qc.sdg([0,1])
        qc.h([0,1])

    qc.measure_all()
    return run_counts(qc)

## Part B — Z, ZZ from one measurement group

In [ ]:
reset_execution_counter()

z_counts = measure_pauli_pair(qc_state, "Z")

z0 = parity_expectation(z_counts, [0])
z1 = parity_expectation(z_counts, [1])
z0z1 = parity_expectation(z_counts, [0,1])

print("<Z0> =", z0)
print("<Z1> =", z1)
print("<Z0Z1> =", z0z1)
plot_histogram(z_counts)

## Part C — X and Y basis measurements

In [ ]:
x_counts = measure_pauli_pair(qc_state, "X")
y_counts = measure_pauli_pair(qc_state, "Y")

x0x1 = parity_expectation(x_counts, [0,1])
y0y1 = parity_expectation(y_counts, [0,1])

print("<X0X1> =", x0x1)
print("<Y0Y1> =", y0y1)

## Part D — Reconstruct an energy

A quantum computer can only tell you `0` or `1` on each wire. When the energy is

$$ E = \langle\psi|H|\psi\rangle,\qquad
H = c_0 I + c_1 Z_0 + c_2 Z_1 + c_3 Z_0Z_1 + c_4 (X_0X_1 + Y_0Y_1) .$$

So: how do you get a *number in Hartree* out of a *pile of bitstrings*? And how many
bitstrings do you need?

1. Turn measurement counts into an expectation value $\langle P\rangle$ for any Pauli string.
2. Reproduce the lecture's numerical example line by line.
3. Measure $\langle X_0X_1\rangle$ and $\langle Y_0Y_1\rangle$ using $H$ and $S^\dagger H$
   basis rotations, on a machine that only measures $Z$.
4. Group commuting terms so 5 Pauli terms need only **3** measurement settings.


In [ ]:
c0 = -1.0523732458
c1 = +0.3979374248
c2 = -0.3979374248
c3 = -0.0112801043
c4 = +0.1809311998

energy = (
    c0
    + c1*z0
    + c2*z1
    + c3*z0z1
    + c4*(x0x1 + y0y1)
)

print("Estimated energy =", energy, "Hartree")

print("Quantum measurement circuits executed:", CIRCUIT_EXECUTIONS)

### YOUR TURN
Change `theta` to `0`, `0.2`, and `0.8`. Record the energy.

## Reflection
Why can \(Z_0\), \(Z_1\), and \(Z_0Z_1\) be estimated from the same shots, while \(X_0X_1\) requires a different circuit?

<details>
<summary><b>Instructor solution / suggested answer</b></summary>


    All Z-type observables use the same local Z measurement basis. \(X_0X_1\) requires rotating the X basis into the computational basis using Hadamard gates before measurement. \(Y_0Y_1\) similarly requires \(S^\dagger H\).

</details>